# 🚌 Suroboyo Bus — P5 Data Engineering Pipeline
**Big Data Project — Institut Teknologi Sepuluh Nopember**

**Tugas P5:**
- Polling bus position dari Klacak API
- Klasifikasi bus: `bergerak` / `mangkal` / `berhenti_rute`
- Hitung headway real (exclude mangkal) vs SPM Dishub 15 menit
- Feature engineering untuk ML model
- Output CSV siap pakai untuk P3/P4

**Dataset statis (Kaggle input):**
| File | Digunakan untuk |
|---|---|
| `Data Armada SuroboyoBus 2025.xlsx` | Enrichment jumlah armada resmi per koridor |
| `Data Halte SuroboyoBus WaraWiri API.xlsx` | Jumlah halte per rute |
| `Data Koridor SuroboyoBus WaraWiri API.xlsx` | Metadata koridor (siklus, tipe) |
| `Halte_Suroboyo_dengan_Koordinat.csv` | Update pool-location dari halte resmi |

**Output files:**
| File | Isi |
|---|---|
| `bus_timeseries.csv` | Raw posisi bus tiap polling |
| `koridor_summary.csv` | Headway + armada per koridor per snapshot |
| `feature_engineered.csv` | Dataset final siap ML |
| `speed_heatmap.csv` | Kecepatan rata-rata per koridor per jam |
| `spatial_distribution.csv` | Distribusi spasial armada |

## ⚙️ 0. Config
> **Ganti `DURATION_SEC`** sebelum run: `120` untuk test, `3600` untuk 1 jam penuh.

In [12]:
import requests, math, time, os, csv
import pandas as pd
from datetime import datetime

# ── Ganti sesuai kebutuhan ──────────────────────────
INTERVAL_SEC  = 30    # poll tiap N detik
DURATION_SEC  = 300  # 120 = test (4 poll) | 3600 = 1 jam (120 poll)
# ────────────────────────────────────────

N_POLLS  = DURATION_SEC // INTERVAL_SEC

BUS_FILE     = "bus_timeseries.csv"
KORIDOR_FILE = "koridor_summary.csv"
FEATURE_FILE = "feature_engineered.csv"

# ── Path dataset statis Kaggle ───────────────────────
KAGGLE_DIR   = "/kaggle/input/datasets/yafiar/suroboyobus"
FILE_ARMADA      = f"{KAGGLE_DIR}/Data Armada SuroboyoBus 2025.xlsx"
FILE_HALTE       = f"{KAGGLE_DIR}/Data Halte SuroboyoBus WaraWiri API.xlsx"
FILE_KORIDOR_META= f"{KAGGLE_DIR}/Data Koridor SuroboyoBus WaraWiri API.xlsx"
FILE_HALTE_COORD = f"{KAGGLE_DIR}/Halte_Suroboyo_dengan_Koordinat.csv"

POOL_RADIUS_M   = 200
SPEED_THRESHOLD = 3    # km/h
SPM_HEADWAY     = 15   # menit — standar pelayanan minimal Dishub Surabaya

print(f"Config loaded — {N_POLLS} polls x {INTERVAL_SEC}s = {DURATION_SEC}s total")

Config loaded — 10 polls x 30s = 300s total


## 📂 0b. Load Dataset Statis Kaggle
Dataset offline digunakan untuk:
1. Memperkaya fitur ML (jumlah armada resmi, jumlah halte, siklus resmi per koridor)
2. Memperbarui daftar pool/terminal dari data halte resmi

In [13]:
# ── Load & inspeksi dataset statis ──────────────────────────────────
print("Loading static datasets...")

df_armada        = pd.read_excel(FILE_ARMADA)
df_halte_raw     = pd.read_excel(FILE_HALTE)
df_koridor_meta  = pd.read_excel(FILE_KORIDOR_META)
df_halte_coord   = pd.read_csv(FILE_HALTE_COORD)

print(f"  Armada          : {len(df_armada)} baris | kolom: {list(df_armada.columns)}")
print(f"  Halte           : {len(df_halte_raw)} baris | kolom: {list(df_halte_raw.columns)}")
print(f"  Koridor meta    : {len(df_koridor_meta)} baris | kolom: {list(df_koridor_meta.columns)}")
print(f"  Halte+Koordinat : {len(df_halte_coord)} baris | kolom: {list(df_halte_coord.columns)}")

df_armada.head(3)

Loading static datasets...
  Armada          : 5 baris | kolom: ['kategori', 'jumlah']
  Halte           : 902 baris | kolom: ['ID', 'Nama_Halte', 'Rute_yang_Melewati']
  Koridor meta    : 15 baris | kolom: ['KEY', 'KODE', 'ID', 'RUTE', 'JAM_OPS', 'KETERANGAN']
  Halte+Koordinat : 111 baris | kolom: ['ID', 'Nama_Halte', 'Rute_yang_Melewati', 'Latitude', 'Longitude']


,kategori,jumlah
0,suroboyo_bus_diesel,28
1,suroboyo_bus_listrik,12
2,trans_semanggi,17


In [14]:
# ── Detect nama kolom kunci secara otomatis ──────────────────────────
def find_col(df, keywords):
    """Return nama kolom pertama yang mengandung salah satu keyword (case-insensitive)."""
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return df.columns[0]  # fallback ke kolom pertama

KOR_CODE_COL  = find_col(df_koridor_meta, ['kode', 'no.', 'no ', 'koridor'])
KOR_CYCLE_COL = find_col(df_koridor_meta, ['siklus', 'cycle', 'waktu'])
KOR_TYPE_COL  = find_col(df_koridor_meta, ['tipe', 'type', 'feeder', 'jenis'])

ARM_CODE_COL  = find_col(df_armada, ['kode', 'no.', 'no ', 'koridor'])
ARM_COUNT_COL = find_col(df_armada, ['jumlah', 'unit', 'armada', 'bus'])

HALTE_CODE_COL = find_col(df_halte_raw, ['kode', 'no.', 'koridor', 'rute'])

print(f"Kode koridor (meta)  : '{KOR_CODE_COL}'")
print(f"Siklus koridor       : '{KOR_CYCLE_COL}'")
print(f"Tipe koridor         : '{KOR_TYPE_COL}'")
print(f"Kode armada          : '{ARM_CODE_COL}'")
print(f"Jumlah armada        : '{ARM_COUNT_COL}'")
print(f"Kode halte           : '{HALTE_CODE_COL}'")

# ── Bangun lookup dicts ──────────────────────────────────────────────
# kode_koridor (str) → jumlah armada resmi
armada_lookup = (
    df_armada.dropna(subset=[ARM_CODE_COL, ARM_COUNT_COL])
    .assign(_k=df_armada[ARM_CODE_COL].astype(str).str.strip())
    .set_index('_k')[ARM_COUNT_COL]
    .to_dict()
)

# kode_koridor (str) → jumlah halte
halte_lookup = (
    df_halte_raw.assign(_k=df_halte_raw[HALTE_CODE_COL].astype(str).str.strip())
    .groupby('_k').size().to_dict()
)

# kode_koridor (str) → siklus resmi (menit)
cycle_lookup = (
    df_koridor_meta.dropna(subset=[KOR_CODE_COL, KOR_CYCLE_COL])
    .assign(_k=df_koridor_meta[KOR_CODE_COL].astype(str).str.strip())
    .set_index('_k')[KOR_CYCLE_COL]
    .to_dict()
) if KOR_CYCLE_COL else {}

print(f"\narmada_lookup  : {len(armada_lookup)} entri, contoh: {dict(list(armada_lookup.items())[:3])}")
print(f"halte_lookup   : {len(halte_lookup)} entri, contoh: {dict(list(halte_lookup.items())[:3])}")
print(f"cycle_lookup   : {len(cycle_lookup)} entri, contoh: {dict(list(cycle_lookup.items())[:3])}")

Kode koridor (meta)  : 'KODE'
Siklus koridor       : 'KEY'
Tipe koridor         : 'KEY'
Kode armada          : 'kategori'
Jumlah armada        : 'jumlah'
Kode halte           : 'Rute_yang_Melewati'

armada_lookup  : 5 entri, contoh: {'suroboyo_bus_diesel': 28, 'suroboyo_bus_listrik': 12, 'trans_semanggi': 17}
halte_lookup   : 69 entri, contoh: {'fd02 2': 42, 'fd02 2 / fd05 5': 1, 'fd02 2 / fd05 5 / fd09 9': 1}
cycle_lookup   : 15 entri, contoh: {'1': 'sbr1', '10': 'tmk2', '51': 'sbr4'}


In [15]:
# ── Bangun POOL_LOCATIONS dari halte resmi berkoordinat ────────────────────
lat_col  = next((c for c in df_halte_coord.columns if 'lat' in c.lower()), None)
lng_col  = next((c for c in df_halte_coord.columns
                 if 'lon' in c.lower() or 'lng' in c.lower()), None)
nama_col = find_col(df_halte_coord, ['nama', 'name', 'halte'])

print(f"Kolom: nama='{nama_col}', lat='{lat_col}', lng='{lng_col}'")

POOL_LOCATIONS_DEFAULT = [
    {"nama": "Terminal Purabaya",    "lat": -7.351352, "lng": 112.725213},
    {"nama": "Terminal Benowo",      "lat": -7.23589,  "lng": 112.60778},
    {"nama": "Terminal Joyoboyo",    "lat": -7.298961, "lng": 112.736182},
    {"nama": "TIJ Intermoda",        "lat": -7.298961, "lng": 112.736182},
    {"nama": "TOW Osowilangun",      "lat": -7.218318, "lng": 112.653038},
    {"nama": "Terminal Bratang",     "lat": -7.29876,  "lng": 112.76159},
    {"nama": "Terminal Manukan",     "lat": -7.2389,   "lng": 112.6814},
    {"nama": "Kejawan Putih Tambak", "lat": -7.276458, "lng": 112.802087},
    {"nama": "Terminal Menanggal",   "lat": -7.34162,  "lng": 112.72259},
    {"nama": "Park and Ride Sungkono","lat":-7.290839, "lng": 112.715194},
    {"nama": "Terminal Keputih",     "lat": -7.29471,  "lng": 112.80176},
]

if lat_col and lng_col:
    POOL_KEYWORDS = ['terminal', 'pool', 'park', 'tij', 'tow', 'intermoda', 'osowilangun']
    mask = df_halte_coord[nama_col].str.lower().str.contains(
        '|'.join(POOL_KEYWORDS), na=False
    )
    df_pools = df_halte_coord[mask].dropna(subset=[lat_col, lng_col])
    if len(df_pools) >= 3:
        POOL_LOCATIONS = [
            {"nama": str(row[nama_col]),
             "lat": float(row[lat_col]),
             "lng": float(row[lng_col])}
            for _, row in df_pools.iterrows()
        ]
        print(f"Pool dari dataset: {len(POOL_LOCATIONS)} titik")
    else:
        POOL_LOCATIONS = POOL_LOCATIONS_DEFAULT
        print(f"Fallback pool hardcoded: {len(POOL_LOCATIONS)} titik")
else:
    POOL_LOCATIONS = POOL_LOCATIONS_DEFAULT
    print(f"Kolom koordinat tidak ada → fallback: {len(POOL_LOCATIONS)} titik")

for p in POOL_LOCATIONS[:5]:
    print(f"  {p['nama']}: ({p['lat']}, {p['lng']})")

Kolom: nama='Nama_Halte', lat='Latitude', lng='Longitude'
Pool dari dataset: 6 titik
  Darmo Park 1: (-7.2795, 112.7335)
  Darmo Park 2: (-7.2795, 112.7335)
  KENJERAN PARK: (-7.2353, 112.7706)
  Terminal Dukuh Kupang: (-7.2825, 112.7185)
  Terminal Joyoboyo: (-7.3194, 112.7372)


## 🔌 1. Bootstrap — Connect ke Klacak API

In [16]:
print("Connecting to Klacak API...")

bootstrap  = requests.get("https://busmapapi.fly.dev/all").json()
api_url    = bootstrap["apiUrl"]

route_data = requests.get(
    "https://raw.githubusercontent.com/DoubleA4/busmapsby/main/routedata.json"
).json()

halte_data = requests.get(
    "https://raw.githubusercontent.com/DoubleA4/busmapsby/main/halte.json"
).json()["halte"]

print(f"Connected!")
print(f"  {len(route_data)} rute tersedia")
print(f"  {len(halte_data)} halte tersedia")
print(f"  API URL: {api_url}")

Connecting to Klacak API...
Connected!
  16 rute tersedia
  927 halte tersedia
  API URL: https://busmapapi.fly.dev


## 🛠️ 2. Helper Functions

In [17]:
def haversine(lat1, lon1, lat2, lon2):
    """Hitung jarak dua koordinat GPS dalam meter."""
    R  = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a  = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))


def classify_bus(bus):
    """
    Klasifikasi status bus:
      bergerak      : speed >= 3 km/h
      mangkal       : diam DAN berada di dekat terminal/pool
      berhenti_rute : diam tapi tidak di terminal
    """
    speed = float(bus.get("speed", 0))
    if speed >= SPEED_THRESHOLD:
        return "bergerak"
    blat = float(bus.get("lat", 0))
    blng = float(bus.get("lng", 0))
    for pool in POOL_LOCATIONS:
        if haversine(blat, blng, pool["lat"], pool["lng"]) <= POOL_RADIUS_M:
            return "mangkal"
    return "berhenti_rute"


def get_req_addr(code):
    """Tentukan endpoint tracking berdasarkan kode koridor."""
    c = int(code)
    if c < 10 or c in (51, 12): return "sbybus"
    if c < 100:                  return "temanbus"
    return "feeder"


def save_csv(df, filepath, write_header):
    """Simpan CSV dengan QUOTE_ALL agar aman dari parsing error."""
    df.to_csv(
        filepath, mode="a",
        header=write_header,
        index=False,
        quoting=csv.QUOTE_ALL,
    )


def read_csv_safe(filepath):
    """Baca CSV dengan toleransi terhadap baris corrupt."""
    return pd.read_csv(
        filepath,
        quoting=csv.QUOTE_ALL,
        on_bad_lines="skip",
        engine="python",
    )

print("Helper functions loaded.")

Helper functions loaded.


## 🚌 3. Fetch Bus Positions

In [18]:
def fetch_all_buses():
    """
    Ambil posisi semua bus aktif dari semua koridor.
    Return: list of dict, satu entry per bus.
    """
    now  = datetime.now()
    ts   = now.strftime("%Y-%m-%d %H:%M:%S")
    hour = now.hour
    dow  = now.strftime("%A")
    rows = []

    for slug, route in route_data.items():
        code      = route["code"]
        token_str = bootstrap.get(code)
        if not token_str:
            continue
        token = token_str.split("/")[1] if "/" in token_str else token_str
        addr  = get_req_addr(code)

        try:
            resp = requests.get(
                f"{api_url}/track/{addr}/{code}",
                headers={
                    "Authorization": f"Bearer {token}",
                    "Cache-Control": "no-cache",
                },
                timeout=5,
            )
            if not resp.ok:
                continue

            for bus in resp.json():
                status = classify_bus(bus)
                rows.append({
                    # Waktu
                    "timestamp":   ts,
                    "hour":        hour,
                    "day_of_week": dow,
                    "is_weekend":  dow in ["Saturday", "Sunday"],
                    "is_peak":     hour in list(range(6, 10)) + list(range(16, 20)),
                    # Rute
                    "route":       slug,
                    "route_name":  route.get("title", slug),
                    "koridor":     code,   # kode numerik koridor (FIX: dipakai di ML)
                    "feeder":      route.get("feeder", False),
                    # Bus
                    "bus_id":      str(bus.get("info", "unknown")),
                    "lat":         float(bus.get("lat", 0)),
                    "lng":         float(bus.get("lng", 0)),
                    "speed":       float(bus.get("speed", 0)),
                    "direction":   float(bus.get("direction", 0)),
                    "status":      status,
                })
        except Exception:
            pass

    return rows

print("fetch_all_buses() defined.")

fetch_all_buses() defined.


## ⏱️ 4. Headway Calculator

In [19]:
def compute_headway(buses_by_route):
    """
    Hitung headway estimasi per koridor.

    FIX (v3):
    - Guard empty buses_by_route: return empty DataFrame dengan schema penuh.
    - Guard empty rows (semua koridor gagal): sama, return empty DataFrame.
    - Kolom 'koridor' di-carry ke output.
    - Siklus dari cycle_lookup jika tersedia.
    - sort_values pakai na_position='last'.
    """
    SCHEMA = [
        "timestamp", "hour", "day_of_week", "is_weekend", "is_peak",
        "route", "route_name", "koridor", "feeder", "cycle_time_min",
        "n_total", "n_mangkal", "n_berhenti_rute", "n_bergerak", "n_efektif",
        "pct_efektif", "headway_raw_min", "headway_real_min",
        "headway_gap_vs_spm", "avg_speed_kmh", "service_level",
        "n_armada_resmi", "n_halte_rute",
    ]

    if not buses_by_route:
        print("⚠️  compute_headway: buses_by_route kosong — return empty DataFrame")
        return pd.DataFrame(columns=SCHEMA)

    rows = []

    for slug, buses in buses_by_route.items():
        if not buses:
            continue
        route  = route_data.get(slug, {})
        code   = str(buses[0]["koridor"]).strip()

        cycle_default = 60 if route.get("feeder") else 120
        try:
            cycle = int(cycle_lookup.get(code, cycle_default))
        except (ValueError, TypeError):
            cycle = cycle_default

        n_total    = len(set(b["bus_id"] for b in buses))
        n_mangkal  = sum(1 for b in buses if b["status"] == "mangkal")
        n_berhenti = sum(1 for b in buses if b["status"] == "berhenti_rute")
        n_bergerak = sum(1 for b in buses if b["status"] == "bergerak")
        n_efektif  = n_total - n_mangkal

        hw_raw  = round(cycle / n_total,   1) if n_total   > 0 else None
        hw_real = round(cycle / n_efektif, 1) if n_efektif > 0 else None
        avg_spd = round(sum(b["speed"] for b in buses) / len(buses), 1) if buses else 0
        pct_eff = round(n_efektif / n_total * 100, 1) if n_total > 0 else 0
        hw_gap  = round(hw_real - SPM_HEADWAY, 1) if hw_real is not None else None

        if hw_real is None:   svc = "tidak_beroperasi"
        elif hw_real <= 15:   svc = "baik"
        elif hw_real <= 25:   svc = "sedang"
        else:                 svc = "buruk"

        n_armada_resmi = armada_lookup.get(code)
        n_halte_rute   = halte_lookup.get(code)

        rows.append({
            "timestamp":          buses[0]["timestamp"],
            "hour":               buses[0]["hour"],
            "day_of_week":        buses[0]["day_of_week"],
            "is_weekend":         buses[0]["is_weekend"],
            "is_peak":            buses[0]["is_peak"],
            "route":              slug,
            "route_name":         route.get("title", slug),
            "koridor":            code,
            "feeder":             route.get("feeder", False),
            "cycle_time_min":     cycle,
            "n_total":            n_total,
            "n_mangkal":          n_mangkal,
            "n_berhenti_rute":    n_berhenti,
            "n_bergerak":         n_bergerak,
            "n_efektif":          n_efektif,
            "pct_efektif":        pct_eff,
            "headway_raw_min":    hw_raw,
            "headway_real_min":   hw_real,
            "headway_gap_vs_spm": hw_gap,
            "avg_speed_kmh":      avg_spd,
            "service_level":      svc,
            "n_armada_resmi":     n_armada_resmi,
            "n_halte_rute":       n_halte_rute,
        })

    if not rows:
        print("⚠️  compute_headway: semua koridor kosong — return empty DataFrame")
        return pd.DataFrame(columns=SCHEMA)

    df = pd.DataFrame(rows)
    return df.sort_values("headway_real_min", na_position="last")

print("compute_headway() defined.")

compute_headway() defined.


## 🔧 5. Feature Engineering

In [20]:
def feature_engineering(df_kor):
    """
    Tambah fitur turunan untuk ML model.
    Input : koridor_summary DataFrame (sudah include kolom 'koridor')
    Output: DataFrame dengan fitur tambahan + label demand_level
    """
    df = df_kor.copy()

    # Konversi tipe numerik
    num_cols = ["hour", "n_total", "n_mangkal", "n_efektif",
                "headway_real_min", "avg_speed_kmh", "pct_efektif",
                "n_armada_resmi", "n_halte_rute"]
    for col in num_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Kategori waktu
    df["time_category"] = pd.cut(
        df["hour"],
        bins   = [0, 6, 9, 12, 16, 20, 24],
        labels = ["dini_hari", "pagi_peak", "siang",
                  "sore_peak", "malam", "tengah_malam"],
        right  = False,
    ).astype(str)

    # Flag inefisiensi armada
    df["ada_mangkal"] = df["n_mangkal"] > 0
    df["pct_mangkal"] = (
        df["n_mangkal"] / df["n_total"].replace(0, 1) * 100
    ).round(1)

    # Flag headway
    df["headway_ok"]     = df["headway_real_min"] <= SPM_HEADWAY
    df["headway_kritis"] = df["headway_real_min"] > 25

    # Utilisasi armada vs data resmi
    if "n_armada_resmi" in df.columns:
        df["utilisasi_armada"] = (
            df["n_total"] / df["n_armada_resmi"].replace(0, pd.NA)
        ).round(3)
    else:
        df["utilisasi_armada"] = pd.NA

    # Label demand — TARGET VARIABLE untuk ML
    df["demand_level"] = df["n_efektif"].apply(
        lambda x: "tinggi" if x >= 12
        else "sedang"      if x >= 7
        else "rendah"
    )

    # FIX: encode kategorik dengan .map() — robust terhadap string 'True'/'False'
    bool_map = {True: 1, False: 0, "True": 1, "False": 0, 1: 1, 0: 0}
    df["feeder_enc"]     = df["feeder"].map(bool_map).fillna(0).astype(int)
    df["is_peak_enc"]    = df["is_peak"].map(bool_map).fillna(0).astype(int)
    df["is_weekend_enc"] = df["is_weekend"].map(bool_map).fillna(0).astype(int)

    return df

print("feature_engineering() defined.")

feature_engineering() defined.


## 👀 6. Snapshot Preview
Cek dulu satu snapshot sebelum mulai polling loop.

In [21]:
print("Fetching snapshot pertama...")
buses = fetch_all_buses()
print(f"{len(buses)} bus terdeteksi\n")

if not buses:
    print("⚠️  Tidak ada data bus — cek koneksi API atau token.")
else:
    by_route = {}
    for b in buses:
        by_route.setdefault(b["route"], []).append(b)

    df_hw = compute_headway(by_route)

    n_mangkal_total = sum(1 for b in buses if b["status"] == "mangkal")
    n_efektif_total = len(buses) - n_mangkal_total

    print("=" * 80)
    print("SNAPSHOT — ARMADA & HEADWAY PER KORIDOR")
    print("=" * 80)
    if df_hw.empty:
        print("DataFrame kosong — tidak ada koridor aktif.")
    else:
        cols = [c for c in ["koridor", "route_name", "n_total", "n_mangkal", "n_efektif",
                             "n_armada_resmi", "headway_raw_min", "headway_real_min",
                             "avg_speed_kmh", "service_level"] if c in df_hw.columns]
        print(df_hw[cols].to_string(index=False))

    print(f"\nTotal bus  : {len(buses)}")
    print(f"Mangkal    : {n_mangkal_total}")
    print(f"Efektif    : {n_efektif_total}")

Fetching snapshot pertama...
136 bus terdeteksi

SNAPSHOT — ARMADA & HEADWAY PER KORIDOR
koridor                        route_name  n_total  n_mangkal  n_efektif n_armada_resmi  headway_raw_min  headway_real_min  avg_speed_kmh service_level
    108                TIJ - Gunung Anyar       14          0         14           None              4.3               4.3           20.4          baik
    122     Term. Menanggal-Term. Manukan       13          0         13           None              4.6               4.6           13.2          baik
    127        Purabaya-ITS-Kenjeran Park        9          0          9           None              6.7               6.7           23.6          baik
     12            Term. Benowo-Tunjungan       16          0         16           None              7.5               7.5           14.2          baik
    105       Mayjend Sungkono-Puspa Raya        8          0          8           None              7.5               7.5           13.0          baik

## 📡 7. Polling Loop
> **Pastikan DURATION_SEC sudah benar di cell Config sebelum run cell ini.**

In [22]:
# Hapus file lama — fresh start
for f in [BUS_FILE, KORIDOR_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"File lama dihapus: {f}")

print(f"\nMulai polling {N_POLLS}x | interval {INTERVAL_SEC}s | total {DURATION_SEC}s\n")

for i in range(N_POLLS):

    # Refresh token tiap 30 menit
    if i > 0 and i % 60 == 0:
        bootstrap = requests.get("https://busmapapi.fly.dev/all").json()
        print(f"   Token refreshed at poll #{i+1}")

    buses = fetch_all_buses()
    if not buses:
        print(f"[{i+1:03d}] No data returned, skipping...")
        time.sleep(INTERVAL_SEC)
        continue

    by_route = {}
    for b in buses:
        by_route.setdefault(b["route"], []).append(b)

    df_buses = pd.DataFrame(buses)
    df_kor   = compute_headway(by_route)

    if df_kor.empty:
        print(f"[{i+1:03d}] compute_headway kosong — skip save")
        time.sleep(INTERVAL_SEC)
        continue

    write_header = (i == 0)
    save_csv(df_buses, BUS_FILE,     write_header)
    save_csv(df_kor,   KORIDOR_FILE, write_header)

    n_mang = sum(1 for b in buses if b["status"] == "mangkal")
    n_eff  = len(buses) - n_mang
    print(
        f"[{i+1:03d}] {buses[0]['timestamp']} | "
        f"{len(buses)} bus | {n_mang} mangkal | "
        f"{n_eff} efektif | {len(by_route)} koridor aktif"
    )

    time.sleep(INTERVAL_SEC)

print("\nPolling selesai!")

File lama dihapus: bus_timeseries.csv
File lama dihapus: koridor_summary.csv

Mulai polling 10x | interval 30s | total 300s

[001] 2026-06-28 05:53:16 | 154 bus | 0 mangkal | 154 efektif | 15 koridor aktif
[002] 2026-06-28 05:53:52 | 154 bus | 0 mangkal | 154 efektif | 15 koridor aktif
[003] 2026-06-28 05:54:30 | 154 bus | 0 mangkal | 154 efektif | 15 koridor aktif
[004] 2026-06-28 05:55:07 | 153 bus | 0 mangkal | 153 efektif | 15 koridor aktif
[005] 2026-06-28 05:55:43 | 153 bus | 0 mangkal | 153 efektif | 15 koridor aktif
[006] 2026-06-28 05:56:20 | 98 bus | 0 mangkal | 98 efektif | 11 koridor aktif
[007] 2026-06-28 05:56:57 | 154 bus | 0 mangkal | 154 efektif | 15 koridor aktif
[008] 2026-06-28 05:57:34 | 154 bus | 0 mangkal | 154 efektif | 15 koridor aktif
[009] 2026-06-28 05:58:10 | 136 bus | 0 mangkal | 136 efektif | 15 koridor aktif
[010] 2026-06-28 05:58:46 | 136 bus | 0 mangkal | 136 efektif | 15 koridor aktif

Polling selesai!


## 📊 8. Analisis Dataset

In [23]:
print("ANALISIS DATASET")
print("=" * 80)

df_ts  = read_csv_safe(BUS_FILE)
df_kor = read_csv_safe(KORIDOR_FILE)

# Konversi tipe
for col in ["speed", "lat", "lng", "hour"]:
    df_ts[col] = pd.to_numeric(df_ts[col], errors="coerce")
for col in ["headway_real_min", "n_total", "n_mangkal", "n_efektif", "avg_speed_kmh"]:
    df_kor[col] = pd.to_numeric(df_kor[col], errors="coerce")

print(f"Records bus      : {len(df_ts):,}")
print(f"Records koridor  : {len(df_kor):,}")
print(f"Timespan         : {df_ts['timestamp'].min()} -> {df_ts['timestamp'].max()}")
print(f"Bus unik         : {df_ts['bus_id'].nunique()}")
print(f"Rute terekam     : {df_ts['route'].nunique()}")

ANALISIS DATASET
Records bus      : 1,446
Records koridor  : 146
Timespan         : 2026-06-28 05:53:16 -> 2026-06-28 05:58:46
Bus unik         : 154
Rute terekam     : 15


In [24]:
# Komposisi status bus per rute
print("KOMPOSISI STATUS BUS PER RUTE:")
status_tbl = (
    df_ts.groupby(["route_name", "status"])
    .size().unstack(fill_value=0)
)
print(status_tbl.to_string())

KOMPOSISI STATUS BUS PER RUTE:
status                             bergerak  berhenti_rute
route_name                                                
Kejawan-UNESA                            74             76
Mayjend Sungkono-Balai Kota              23             27
Mayjend Sungkono-Puspa Raya              51             21
Purabaya-ITS-Kenjeran Park               44             44
Purabaya-Perak                           34             76
Purabaya-UNAIR Kampus C                  89             51
SIER-Kota Lama                           34             30
TIJ - Gunung Anyar                      112             14
TIJ - Lakarsantri                       111             51
TOW - UNESA                              18             22
Term. Benowo-Tunjungan                   86             58
Term. Bratang - Stasiun Psr. Turi        26             36
Term. Keputih-Bunguran                   37             23
Term. Menanggal-Term. Manukan            69             59
Terminal Bratang - Shelte

In [25]:
# Speed heatmap (bus bergerak saja)
print("KECEPATAN RATA-RATA BUS BERGERAK PER KORIDOR PER JAM (km/h):")
df_moving   = df_ts[df_ts["status"] == "bergerak"]
speed_pivot = (
    df_moving.groupby(["route_name", "hour"])["speed"]
    .mean().round(1).unstack()
)
print(speed_pivot.to_string())
speed_pivot.reset_index().to_csv("speed_heatmap.csv", index=False, quoting=csv.QUOTE_ALL)
print("\nDisimpan: speed_heatmap.csv")

KECEPATAN RATA-RATA BUS BERGERAK PER KORIDOR PER JAM (km/h):
hour                                  5
route_name                             
Kejawan-UNESA                      21.7
Mayjend Sungkono-Balai Kota        20.9
Mayjend Sungkono-Puspa Raya        25.4
Purabaya-ITS-Kenjeran Park         42.1
Purabaya-Perak                     22.0
Purabaya-UNAIR Kampus C            26.5
SIER-Kota Lama                     25.1
TIJ - Gunung Anyar                 22.8
TIJ - Lakarsantri                  20.8
TOW - UNESA                        39.9
Term. Benowo-Tunjungan             23.3
Term. Bratang - Stasiun Psr. Turi  23.3
Term. Keputih-Bunguran             25.9
Term. Menanggal-Term. Manukan      22.9
Terminal Bratang - Shelter Bulak   25.1

Disimpan: speed_heatmap.csv


In [26]:
# Headway stats per koridor
print("HEADWAY REAL PER KORIDOR (menit):")
hw_stats = (
    df_kor.groupby("route_name")["headway_real_min"]
    .agg(["mean", "min", "max"]).round(1)
)
hw_stats.columns = ["avg_headway", "best_headway", "worst_headway"]
# FIX: lambda cek NaN agar tidak crash pada headway None
hw_stats["vs_spm"] = hw_stats["avg_headway"].apply(
    lambda x: "N/A" if pd.isna(x) else ("OK" if x <= 15 else f"+{round(x-15,1)} mnt")
)
print(hw_stats.sort_values("avg_headway", na_position="last").to_string())

HEADWAY REAL PER KORIDOR (menit):
                                   avg_headway  best_headway  worst_headway    vs_spm
route_name                                                                           
TIJ - Lakarsantri                          3.3           3.3            3.3        OK
TIJ - Gunung Anyar                         4.3           4.3            4.3        OK
Term. Menanggal-Term. Manukan              4.7           4.6            5.0        OK
Purabaya-ITS-Kenjeran Park                 6.9           6.7            7.5        OK
Mayjend Sungkono-Puspa Raya                7.5           7.5            7.5        OK
Term. Benowo-Tunjungan                     7.5           7.5            7.5        OK
Kejawan-UNESA                              8.0           8.0            8.0        OK
Purabaya-UNAIR Kampus C                    8.6           8.6            8.6        OK
SIER-Kota Lama                             9.6           8.6           12.0        OK
Term. Keputih-Bungur

In [27]:
# Distribusi spasial (bus efektif saja)
df_eff  = df_ts[df_ts["status"] != "mangkal"]
spatial = df_eff.groupby("route_name").agg(
    lat_min  =("lat",   "min"),
    lat_max  =("lat",   "max"),
    lng_min  =("lng",   "min"),
    lng_max  =("lng",   "max"),
    avg_speed=("speed", "mean"),
).round(4)
spatial.to_csv("spatial_distribution.csv", quoting=csv.QUOTE_ALL)
print("Disimpan: spatial_distribution.csv")
print(spatial.to_string())

Disimpan: spatial_distribution.csv
                                   lat_min  lat_max   lng_min   lng_max  avg_speed
route_name                                                                        
Kejawan-UNESA                      -7.2911  -7.2641  112.6762  112.8026    10.6933
Mayjend Sungkono-Balai Kota        -7.2916  -7.2608  112.7139  112.7505     9.6000
Mayjend Sungkono-Puspa Raya        -7.2950  -7.2562  112.6349  112.7189    17.9722
Purabaya-ITS-Kenjeran Park         -7.3518  -7.2501  112.7127  112.7906    21.0341
Purabaya-Perak                     -7.3533  -7.2037  112.7247  112.7416     6.7909
Purabaya-UNAIR Kampus C            -7.3555  -7.2637  112.7125  112.7894    16.8286
SIER-Kota Lama                     -7.3200  -7.2385  112.7344  112.7508    13.3281
TIJ - Gunung Anyar                 -7.3422  -7.2933  112.7335  112.8089    20.2857
TIJ - Lakarsantri                  -7.3140  -7.2876  112.6337  112.7395    14.2840
TOW - UNESA                        -7.2857  -7.2184 

## 🔧 9. Feature Engineering & Export

In [28]:
df_feat = feature_engineering(df_kor)
save_csv(df_feat, FEATURE_FILE, write_header=True)

print("DISTRIBUSI DEMAND LEVEL (target variable ML):")
print(df_feat["demand_level"].value_counts().to_string())

print("\nDISTRIBUSI SERVICE LEVEL:")
print(df_feat["service_level"].value_counts().to_string())

print("\nDISTRIBUSI TIME CATEGORY:")
print(df_feat["time_category"].value_counts().to_string())

DISTRIBUSI DEMAND LEVEL (target variable ML):
demand_level
tinggi    65
rendah    48
sedang    33

DISTRIBUSI SERVICE LEVEL:
service_level
baik      142
sedang      2
buruk       2

DISTRIBUSI TIME CATEGORY:
time_category
dini_hari    146


In [29]:
print("OUTPUT FILES:")
print("-" * 60)
for f in [BUS_FILE, KORIDOR_FILE, FEATURE_FILE,
          "speed_heatmap.csv", "spatial_distribution.csv"]:
    if os.path.exists(f):
        n_rows = len(read_csv_safe(f))
        size   = os.path.getsize(f) // 1024
        print(f"  {f:<40} {n_rows:>6} rows | {size} KB")

print("\nKOLOM FEATURE_ENGINEERED.CSV (input ML):")
ml_cols = [
    "koridor", "hour", "is_peak_enc", "is_weekend_enc", "feeder_enc",
    "time_category", "n_total", "n_efektif", "pct_efektif",
    "headway_real_min", "headway_gap_vs_spm", "headway_ok",
    "avg_speed_kmh", "ada_mangkal", "pct_mangkal",
    "n_armada_resmi", "n_halte_rute", "utilisasi_armada",
    "service_level", "demand_level",
]
for c in ml_cols:
    dtype = str(df_feat[c].dtype) if c in df_feat.columns else "MISSING"
    print(f"  {c:<30} {dtype}")

print("\nPipeline P5 selesai!")

OUTPUT FILES:
------------------------------------------------------------
  bus_timeseries.csv                         1446 rows | 227 KB
  koridor_summary.csv                         146 rows | 24 KB
  feature_engineered.csv                      146 rows | 33 KB
  speed_heatmap.csv                            15 rows | 0 KB
  spatial_distribution.csv                     15 rows | 1 KB

KOLOM FEATURE_ENGINEERED.CSV (input ML):
  koridor                        int64
  hour                           int64
  is_peak_enc                    int64
  is_weekend_enc                 int64
  feeder_enc                     int64
  time_category                  object
  n_total                        int64
  n_efektif                      int64
  pct_efektif                    float64
  headway_real_min               float64
  headway_gap_vs_spm             float64
  headway_ok                     bool
  avg_speed_kmh                  float64
  ada_mangkal                    bool
  pct_mangkal   

## 🤖 10. ML Model — Rekomendasi Nomor Bus
Dua model untuk memprediksi demand level koridor dan merekomendasikan prioritas nomor bus.

### Akar Masalah Overfitting (dan Solusinya)
| Masalah | Solusi |
|---|---|
| **Data Leakage**: `n_efektif`, `headway_real_min` secara matematis mendefinisikan `demand_level` | Gunakan hanya fitur non-leaky |
| **Route Memorization**: random split | **GroupKFold** by koridor |
| Model terlalu kompleks | Regularisasi XGBoost + LSTM Dropout + EarlyStopping |

### Dua Model
1. **XGBoost** — klasifikasi dari fitur tabular (kontekstual + proxy + dataset statis)
2. **LSTM** — prediksi dari urutan time series per koridor

In [30]:
# Install dependensi jika belum ada
import subprocess, sys
for _pkg in ["xgboost", "tensorflow"]:
    try:
        __import__(_pkg)
    except ImportError:
        print(f"Installing {_pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"])

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score

# ── Load & bersihkan feature-engineered data ─────────────────────────────
df_ml = read_csv_safe(FEATURE_FILE)

NUM_COLS = ["hour", "n_total", "n_mangkal", "n_efektif", "n_bergerak",
            "cycle_time_min", "headway_real_min", "headway_gap_vs_spm",
            "avg_speed_kmh", "pct_efektif", "pct_mangkal",
            "is_peak_enc", "is_weekend_enc", "feeder_enc",
            "n_armada_resmi", "n_halte_rute", "utilisasi_armada"]
for c in NUM_COLS:
    if c in df_ml.columns:
        df_ml[c] = pd.to_numeric(df_ml[c], errors="coerce")

df_ml["ada_mangkal"] = (
    df_ml["ada_mangkal"]
    .map({"True": 1, "False": 0, True: 1, False: 0})
    .fillna(0).astype(int)
)
df_ml = df_ml.dropna(subset=["demand_level", "route_name", "headway_real_min"])
df_ml = df_ml.sort_values("timestamp").reset_index(drop=True)

# FIX: defensive check kolom 'koridor'
if "koridor" not in df_ml.columns:
    df_ml["koridor"] = df_ml.get("route", df_ml.index.astype(str))
    print("⚠️  Kolom 'koridor' tidak ditemukan — menggunakan 'route' sebagai fallback")

le_demand = LabelEncoder()
df_ml["y"] = le_demand.fit_transform(df_ml["demand_level"])
le_route   = LabelEncoder()
df_ml["route_enc"] = le_route.fit_transform(df_ml["route_name"])

print(f"Dataset  : {len(df_ml)} baris, {df_ml['route_name'].nunique()} rute unik")
print(f"Kelas ML : {list(le_demand.classes_)}")
print("\nDistribusi demand_level:")
print(df_ml["demand_level"].value_counts().to_string())

2026-06-28 05:59:26.954789: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782626367.199244      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782626367.271053      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782626367.842697      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782626367.842761      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782626367.842763      58 computation_placer.cc:177] computation placer alr

Dataset  : 146 baris, 15 rute unik
Kelas ML : ['rendah', 'sedang', 'tinggi']

Distribusi demand_level:
demand_level
tinggi    65
rendah    48
sedang    33


In [31]:
# ── FITUR NON-LEAKY ────────────────────────────────────────────────────────────
# Tidak dimasukkan: n_efektif, pct_efektif, headway_real_min (data leakage)
# Fitur baru dari dataset statis: utilisasi_armada, n_halte_rute
ALL_SAFE = [
    "hour",             # jam polling
    "is_peak_enc",      # jam sibuk
    "is_weekend_enc",   # akhir pekan
    "feeder_enc",       # tipe rute
    "cycle_time_min",   # siklus resmi
    "avg_speed_kmh",    # proxy kemacetan
    "ada_mangkal",      # ada bus nganggur
    "pct_mangkal",      # % bus mangkal
    "utilisasi_armada", # n_total / n_armada_resmi
    "n_halte_rute",     # proxy panjang/kompleksitas rute
]
# Hanya pakai fitur yang benar-benar ada di df_ml
SAFE_FEATURES = [f for f in ALL_SAFE if f in df_ml.columns]
print(f"Fitur yang digunakan ({len(SAFE_FEATURES)}): {SAFE_FEATURES}")

X      = df_ml[SAFE_FEATURES].fillna(0).values
y      = df_ml["y"].values
groups = df_ml["route_enc"].values

# ── XGBoost dengan regularisasi ─────────────────────────────────────────────
xgb_clf = xgb.XGBClassifier(
    n_estimators     = 200,
    max_depth        = 3,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.5,
    reg_lambda       = 2.0,
    min_child_weight = 10,
    eval_metric      = "mlogloss",
    # FIX: hapus use_label_encoder (deprecated di XGBoost >= 1.6)
    random_state     = 42,
    verbosity        = 0,
)

gkf   = GroupKFold(n_splits=min(5, df_ml["route_enc"].nunique()))
cv_f1 = cross_val_score(xgb_clf, X, y, groups=groups, cv=gkf, scoring="f1_weighted")
xgb_clf.fit(X, y)
train_f1 = f1_score(y, xgb_clf.predict(X), average="weighted")

print("=" * 60)
print("XGBOOST — EVALUASI ANTI-OVERFIT")
print("=" * 60)
print(f"Train F1 (weighted)   : {train_f1:.3f}")
print(f"GroupKFold CV F1      : {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")
print(f"Skor per fold         : {[round(s, 3) for s in cv_f1]}")
gap_xgb = train_f1 - cv_f1.mean()
print(f"Overfitting gap       : {gap_xgb:.3f}  →  "
      f"{'OVERFITTING' if gap_xgb > 0.15 else 'Terkontrol ✓'}")

print("\nFEATURE IMPORTANCE:")
fi = pd.Series(xgb_clf.feature_importances_, index=SAFE_FEATURES)
print(fi.sort_values(ascending=False).round(3).to_string())

print("\nCLASSIFICATION REPORT (train):")
print(classification_report(y, xgb_clf.predict(X), target_names=le_demand.classes_))

Fitur yang digunakan (10): ['hour', 'is_peak_enc', 'is_weekend_enc', 'feeder_enc', 'cycle_time_min', 'avg_speed_kmh', 'ada_mangkal', 'pct_mangkal', 'utilisasi_armada', 'n_halte_rute']
XGBOOST — EVALUASI ANTI-OVERFIT
Train F1 (weighted)   : 0.708
GroupKFold CV F1      : 0.264 ± 0.180
Skor per fold         : [np.float64(0.334), np.float64(0.078), np.float64(0.513), np.float64(0.036), np.float64(0.359)]
Overfitting gap       : 0.444  →  OVERFITTING

FEATURE IMPORTANCE:
feeder_enc          0.550
cycle_time_min      0.406
avg_speed_kmh       0.044
hour                0.000
is_peak_enc         0.000
is_weekend_enc      0.000
ada_mangkal         0.000
pct_mangkal         0.000
utilisasi_armada    0.000
n_halte_rute        0.000

CLASSIFICATION REPORT (train):
              precision    recall  f1-score   support

      rendah       0.56      0.85      0.68        48
      sedang       0.92      0.33      0.49        33
      tinggi       0.87      0.82      0.84        65

    accuracy       

In [32]:
# ── LSTM ───────────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Hanya pakai LSTM_FEATURES yang ada di df_ml
ALL_LSTM = ["avg_speed_kmh", "pct_mangkal", "is_peak_enc", "feeder_enc", "utilisasi_armada"]
LSTM_FEATURES = [f for f in ALL_LSTM if f in df_ml.columns]
WINDOW        = 5
print(f"LSTM features ({len(LSTM_FEATURES)}): {LSTM_FEATURES}")

seqs_X, seqs_y = [], []
for route_id, grp in df_ml.groupby("route_enc"):
    grp  = grp.sort_values("timestamp").reset_index(drop=True)
    vals = grp[LSTM_FEATURES].fillna(0).values.astype("float32")
    labs = grp["y"].values
    if len(grp) <= WINDOW:
        continue
    for t in range(WINDOW, len(grp)):
        seqs_X.append(vals[t - WINDOW : t])
        seqs_y.append(labs[t])

X_lstm = np.array(seqs_X, dtype="float32")
y_lstm = np.array(seqs_y, dtype="int32")
n_cls  = len(le_demand.classes_)

# Guard: skip LSTM jika data terlalu sedikit
if len(X_lstm) < 20:
    print(f"⚠️  Terlalu sedikit LSTM sequences ({len(X_lstm)}) — skip LSTM training")
    val_acc = 0.0; gap_lstm = 0.0; lstm_model = None
else:
    rng   = np.random.RandomState(42)
    idx   = rng.permutation(len(X_lstm))
    split = int(0.8 * len(idx))
    X_tr, X_val = X_lstm[idx[:split]], X_lstm[idx[split:]]
    y_tr, y_val = y_lstm[idx[:split]], y_lstm[idx[split:]]
    print(f"LSTM sequences — train: {len(X_tr)}, val: {len(X_val)}, shape: {X_lstm.shape}")

    tf.random.set_seed(42)
    lstm_model = keras.Sequential([
        layers.LSTM(64, return_sequences=True,
                    input_shape=(WINDOW, len(LSTM_FEATURES)),
                    kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.3),
        layers.LSTM(32, kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(n_cls, activation="softmax"),
    ], name="lstm_bus_demand")

    lstm_model.compile(
        optimizer = keras.optimizers.Adam(1e-3),
        loss      = "sparse_categorical_crossentropy",
        metrics   = ["accuracy"],
    )
    lstm_model.summary()

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=10, restore_best_weights=True, verbose=0
    )
    history = lstm_model.fit(
        X_tr, y_tr,
        validation_data = (X_val, y_val),
        epochs=80, batch_size=32,
        callbacks=[early_stop], verbose=0,
    )
    n_epochs = len(history.history["loss"])
    tr_acc   = history.history["accuracy"][-1]
    val_acc  = history.history["val_accuracy"][-1]
    gap_lstm = tr_acc - val_acc

    print(f"\nLSTM berhenti di epoch {n_epochs} (patience=10)")
    print(f"Train loss / accuracy : {history.history['loss'][-1]:.4f} / {tr_acc:.3f}")
    print(f"Val   loss / accuracy : {history.history['val_loss'][-1]:.4f} / {val_acc:.3f}")
    print(f"Overfitting gap       : {gap_lstm:.3f}  →  "
          f"{'OVERFITTING' if gap_lstm > 0.15 else 'Terkontrol ✓'}")

LSTM features (5): ['avg_speed_kmh', 'pct_mangkal', 'is_peak_enc', 'feeder_enc', 'utilisasi_armada']
LSTM sequences — train: 56, val: 15, shape: (71, 5, 5)


I0000 00:00:1782626383.049537      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782626383.055967      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "lstm_bus_demand"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 5, 64)          │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,435 (118.89 KB)

 Trainable params: 30,435 (118.89 KB)

 Non-trainable params: 0 (0.00 B)

I0000 00:00:1782626387.287067     138 cuda_dnn.cc:529] Loaded cuDNN version 91002



LSTM berhenti di epoch 11 (patience=10)
Train loss / accuracy : 1.0438 / 0.500
Val   loss / accuracy : 1.1275 / 0.267
Overfitting gap       : 0.233  →  OVERFITTING


In [34]:
# ── REKOMENDASI NOMOR BUS — Snapshot Terbaru ────────────────────────────────
# FIX: kolom 'koridor' kini ada di df_ml karena compute_headway() sudah carry-through

latest_ts = df_ml["timestamp"].max()
df_rec    = df_ml[df_ml["timestamp"] == latest_ts].copy()
X_rec     = df_rec[SAFE_FEATURES].fillna(0).values

# XGBoost prediction
df_rec["pred_xgb"] = le_demand.inverse_transform(xgb_clf.predict(X_rec))
proba_xgb          = xgb_clf.predict_proba(X_rec)
df_rec["conf_xgb"] = proba_xgb.max(axis=1).round(2)

# LSTM prediction
lstm_preds = {}
if lstm_model is not None:
    for route_id, grp in df_ml.groupby("route_enc"):
        grp = grp.sort_values("timestamp").reset_index(drop=True)
        if len(grp) < WINDOW:
            continue
        window = grp[LSTM_FEATURES].fillna(0).values[-WINDOW:].astype("float32")
        p      = lstm_model.predict(window[np.newaxis], verbose=0)[0]
        lstm_preds[route_id] = le_demand.classes_[p.argmax()]

df_rec["pred_lstm"] = df_rec["route_enc"].map(lstm_preds).fillna("-")

PRIORITY = {
    "tinggi": "🔴 PRIORITAS — tambah armada",
    "sedang": "🟡 Normal — pantau headway",
    "rendah": "🟢 Efisiensi — pertimbangkan redistribusi",
}
df_rec["rekomendasi"] = df_rec["pred_xgb"].map(PRIORITY)

print(f"REKOMENDASI NOMOR BUS — {latest_ts}")
print("=" * 90)
# FIX: 'koridor' sekarang ada — tidak akan KeyError lagi
out = df_rec[["koridor", "route_name", "pred_xgb", "pred_lstm", "conf_xgb", "rekomendasi"]].copy()
out.columns = ["No.Bus", "Nama Rute", "XGBoost", "LSTM", "Conf.", "Rekomendasi"]
print(out.sort_values("XGBoost", ascending=False).to_string(index=False))

print("\n" + "-" * 50)
print(f"XGBoost CV F1 (GroupKFold) : {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")
print(f"LSTM Val Accuracy           : {val_acc:.3f}")
print(f"Overfitting XGBoost         : {gap_xgb:.3f}  →  "
      f"{'OVERFITTING' if gap_xgb > 0.15 else 'OK ✓'}")
print(f"Overfitting LSTM            : {gap_lstm:.3f}  →  "
      f"{'OVERFITTING' if gap_lstm > 0.15 else 'OK ✓'}")

REKOMENDASI NOMOR BUS — 2026-06-28 05:58:46
 No.Bus                         Nama Rute XGBoost   LSTM  Conf.                              Rekomendasi
    121                    SIER-Kota Lama  tinggi tinggi   0.46              🔴 PRIORITAS — tambah armada
    102       Mayjend Sungkono-Balai Kota  tinggi tinggi   0.48              🔴 PRIORITAS — tambah armada
     51           Purabaya-UNAIR Kampus C  tinggi tinggi   0.88              🔴 PRIORITAS — tambah armada
     12            Term. Benowo-Tunjungan  tinggi tinggi   0.72              🔴 PRIORITAS — tambah armada
    122     Term. Menanggal-Term. Manukan  tinggi tinggi   0.42              🔴 PRIORITAS — tambah armada
    106                 TIJ - Lakarsantri  tinggi tinggi   0.52              🔴 PRIORITAS — tambah armada
     10                     Kejawan-UNESA  tinggi tinggi   0.84              🔴 PRIORITAS — tambah armada
      1                    Purabaya-Perak  tinggi tinggi   0.83              🔴 PRIORITAS — tambah armada
    127    